# 01 — Data Acquisition

**FinPulse Project — Gold & Commodities Trading Pipeline**

Questo notebook copre la **Fase 1** del progetto: acquisizione dati da fonti eterogenee.

| Requisito (FAQ 5) | Come lo copriamo |
|---|---|
| Almeno 2 fonti diverse | Yahoo Finance API + scraping news |
| Almeno 1 API o scraping | Entrambi: API (`yfinance`) + scraping (`BeautifulSoup`) |

**Fonti dati:**
1. **Yahoo Finance API** (`yfinance`) — prezzi OHLCV per oro, argento, DXY, S&P500
2. **Web Scraping** (`BeautifulSoup`) — headlines di news finanziarie
3. **Storage** — tutti i dati grezzi vengono salvati in MongoDB (`finpulse` database)

**Nota Docker:** i servizi si raggiungono tramite i nomi definiti nel `docker-compose.yml`:
- MongoDB → `mongo:27017` (NON `localhost`)
- Neo4j → `neo4j:7687`
- Kafka → `kafka:29092`

## 0. Setup e dipendenze

In [21]:
# Installazione dipendenze (eseguire solo la prima volta)
!pip install yfinance pymongo requests beautifulsoup4 lxml pandas plotly -q

In [22]:
import os
import json
import time
from datetime import datetime, timedelta

import requests
import pandas as pd
import yfinance as yf
from bs4 import BeautifulSoup
from pymongo import MongoClient

print(f"Setup completato — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Setup completato — 2026-03-28 15:31:00


## 1. Configurazione

I parametri vengono letti dalle variabili d'ambiente impostate nel `docker-compose.yml`.
Se sei dentro il container Jupyter, le env var puntano ai service names Docker.
Se lavori fuori da Docker, i default puntano a `localhost`.

In [23]:
# ============================================================
# CONFIGURAZIONE — Docker-aware
# Dentro Docker: le env var sono settate dal docker-compose.yml
#   MONGO_URI=mongodb://mongo:27017
# Fuori Docker: fallback a localhost
# ============================================================

MONGO_URI = os.getenv("MONGO_URI", "mongodb://mongo:27017")
MONGO_DB  = "finpulse"

# Asset da monitorare (Yahoo Finance tickers)
ASSETS = {
    "gold":       "GC=F",       # Gold Futures
    "silver":     "SI=F",       # Silver Futures
    "dxy":        "DX-Y.NYB",   # US Dollar Index (inversamente correlato all'oro)
    "sp500":      "^GSPC",      # S&P 500 (per confronto)
}

# Parametri di acquisizione
HISTORICAL_PERIOD = "2y"   # 2 anni di dati storici
CANDLE_INTERVAL   = "1d"   # candele giornaliere (per storico; per real-time useremo 1h)

# Scraping headers
SCRAPING_HEADERS = {
    "User-Agent": "Mozilla/5.0 (educational-project-finpulse)"
}

print(f"MONGO_URI: {MONGO_URI}")
print(f"Assets:    {list(ASSETS.keys())}")
print(f"Periodo:   {HISTORICAL_PERIOD}, Intervallo: {CANDLE_INTERVAL}")

MONGO_URI: mongodb://mongo:27017
Assets:    ['gold', 'silver', 'dxy', 'sp500']
Periodo:   2y, Intervallo: 1d


## 2. Connessione a MongoDB

Verifichiamo la connessione al database e creiamo le collection necessarie.

**Schema delle collection:**
- `prices` — candele OHLCV per ogni asset
- `news` — headlines scrappate con metadata
- `acquisition_log` — log di ogni acquisizione per tracciabilità

In [24]:
# Connessione a MongoDB
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)

# Test connessione
try:
    client.admin.command("ping")
    print(f"Connesso a MongoDB: {MONGO_URI}")
    print(f"Database disponibili: {client.list_database_names()}")
except Exception as e:
    print(f"ERRORE connessione MongoDB: {e}")
    print("Assicurati che il container mongo sia avviato (docker compose up -d)")
    raise

# Selezione database e collection
db = client[MONGO_DB]
prices_coll  = db["prices"]
news_coll    = db["news"]
log_coll     = db["acquisition_log"]

print(f"\nDatabase: {MONGO_DB}")
print(f"Collection esistenti: {db.list_collection_names()}")

Connesso a MongoDB: mongodb://mongo:27017
Database disponibili: ['admin', 'config', 'finpulse', 'local', 'macro_finance_dm', 'mydb']

Database: finpulse
Collection esistenti: ['integration_errors', 'prices', 'analysis_results', 'integrated_data', 'quality_issues', 'news', 'quality_reports', 'acquisition_log']


## 3. Fonte 1 — Yahoo Finance API (prezzi OHLCV)

Usiamo la libreria `yfinance` per scaricare dati storici OHLCV.
Questa è la fonte **API** richiesta dalla FAQ 5.

`yfinance` internamente usa `requests` per fare chiamate HTTP all'API di Yahoo Finance,
esattamente come mostrato a lezione con la libreria `requests`.

In [25]:
def fetch_asset_history(ticker_symbol: str, asset_name: str,
                        period: str = "2y", interval: str = "1d") -> pd.DataFrame:
    """
    Scarica i dati storici OHLCV da Yahoo Finance.
    
    Parameters
    ----------
    ticker_symbol : str
        Simbolo Yahoo Finance (es. 'GC=F' per l'oro)
    asset_name : str
        Nome leggibile dell'asset (es. 'gold')
    period : str
        Periodo storico da scaricare (es. '2y', '1y', '6mo')
    interval : str
        Intervallo candele (es. '1d', '1h', '5m')
    
    Returns
    -------
    pd.DataFrame
        DataFrame con colonne: Date, Open, High, Low, Close, Volume, asset_name
    """
    print(f"  Scaricando {asset_name} ({ticker_symbol})...", end=" ")
    
    try:
        ticker = yf.Ticker(ticker_symbol)
        df = ticker.history(period=period, interval=interval)
        
        if df.empty:
            print(f"ATTENZIONE: nessun dato ricevuto")
            return pd.DataFrame()
        
        # Pulizia: rimuovi colonne non necessarie, resetta index
        df = df.drop(columns=["Dividends", "Stock Splits"], errors="ignore")
        df = df.reset_index()
        
        # Aggiungi metadata
        df["asset_name"] = asset_name
        df["ticker"]     = ticker_symbol
        
        # Normalizza il nome della colonna data
        if "Datetime" in df.columns:
            df = df.rename(columns={"Datetime": "Date"})
        
        # Converti timezone-aware datetime a timezone-naive per MongoDB
        if df["Date"].dt.tz is not None:
            df["Date"] = df["Date"].dt.tz_localize(None)
        
        print(f"{len(df)} candele scaricate ({df['Date'].min().date()} → {df['Date'].max().date()})")
        return df
    
    except Exception as e:
        print(f"ERRORE: {e}")
        return pd.DataFrame()

In [26]:
# Scarica dati storici per tutti gli asset
print(f"=== ACQUISIZIONE DATI STORICI ===")
print(f"Periodo: {HISTORICAL_PERIOD}, Intervallo: {CANDLE_INTERVAL}")
print(f"{'='*50}")

all_prices = {}

for asset_name, ticker in ASSETS.items():
    df = fetch_asset_history(ticker, asset_name, HISTORICAL_PERIOD, CANDLE_INTERVAL)
    if not df.empty:
        all_prices[asset_name] = df

print(f"\n{'='*50}")
print(f"Asset scaricati con successo: {len(all_prices)}/{len(ASSETS)}")

=== ACQUISIZIONE DATI STORICI ===
Periodo: 2y, Intervallo: 1d
  Scaricando gold (GC=F)... 503 candele scaricate (2024-03-28 → 2026-03-27)
  Scaricando silver (SI=F)... 503 candele scaricate (2024-03-28 → 2026-03-27)
  Scaricando dxy (DX-Y.NYB)... 503 candele scaricate (2024-03-28 → 2026-03-27)
  Scaricando sp500 (^GSPC)... 501 candele scaricate (2024-03-28 → 2026-03-27)

Asset scaricati con successo: 4/4


In [27]:
# Ispezione rapida dei dati scaricati
for name, df in all_prices.items():
    print(f"\n--- {name.upper()} ---")
    print(f"Shape: {df.shape}")
    print(f"Colonne: {list(df.columns)}")
    print(f"Tipi:")
    print(df.dtypes)
    print(f"\nPrime 3 righe:")
    display(df.head(3))
    print(f"\nUltime 3 righe:")
    display(df.tail(3))


--- GOLD ---
Shape: (503, 8)
Colonne: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'asset_name', 'ticker']
Tipi:
Date          datetime64[s]
Open                float64
High                float64
Low                 float64
Close               float64
Volume                int64
asset_name              str
ticker                  str
dtype: object

Prime 3 righe:


,Date,Open,High,Low,Close,Volume,asset_name,ticker
0,2024-03-28,2193.600098,2234.100098,2187.100098,2217.399902,2040,gold,GC=F
1,2024-04-01,2235.699951,2264.199951,2230.000000,2236.500000,400,gold,GC=F
2,2024-04-02,2252.000000,2279.199951,2247.600098,2261.000000,435,gold,GC=F



Ultime 3 righe:


,Date,Open,High,Low,Close,Volume,asset_name,ticker
500,2026-03-25,4549.0,4551.899902,4541.799805,4549.799805,388,gold,GC=F
501,2026-03-26,4441.5,4443.100098,4375.500000,4375.500000,1070,gold,GC=F
502,2026-03-27,4492.0,4492.000000,4492.000000,4492.000000,1070,gold,GC=F



--- SILVER ---
Shape: (503, 8)
Colonne: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'asset_name', 'ticker']
Tipi:
Date          datetime64[s]
Open                float64
High                float64
Low                 float64
Close               float64
Volume                int64
asset_name              str
ticker                  str
dtype: object

Prime 3 righe:


...

,Date,Open,High,Low,Close,Volume,asset_name,ticker
500,2026-03-25,99.199997,99.669998,99.070000,99.599998,0,dxy,DX-Y.NYB
501,2026-03-26,99.629997,100.010002,99.559998,99.900002,0,dxy,DX-Y.NYB
502,2026-03-27,99.870003,100.209999,99.800003,100.150002,0,dxy,DX-Y.NYB



--- SP500 ---
Shape: (501, 8)
Colonne: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'asset_name', 'ticker']
Tipi:
Date          datetime64[s]
Open                float64
High                float64
Low                 float64
Close               float64
Volume                int64
asset_name              str
ticker                  str
dtype: object

Prime 3 righe:


,Date,Open,High,Low,Close,Volume,asset_name,ticker
0,2024-03-28,5248.029785,5264.850098,5245.819824,5254.350098,3998270000,sp500,^GSPC
1,2024-04-01,5257.970215,5263.950195,5229.200195,5243.770020,3325930000,sp500,^GSPC
2,2024-04-02,5204.290039,5208.339844,5184.049805,5205.810059,3886590000,sp500,^GSPC



Ultime 3 righe:


,Date,Open,High,Low,Close,Volume,asset_name,ticker
498,2026-03-25,6598.350098,6633.939941,6568.410156,6591.899902,4936720000,sp500,^GSPC
499,2026-03-26,6555.859863,6573.220215,6473.790039,6477.160156,4845560000,sp500,^GSPC
500,2026-03-27,6453.890137,6453.890137,6356.080078,6368.850098,5303490000,sp500,^GSPC


### 3.1 Salvataggio prezzi in MongoDB

Salviamo i dati nella collection `prices`.
Ogni documento rappresenta una singola candela OHLCV.

**Schema documento:**
```json
{
    "asset_name": "gold",
    "ticker": "GC=F",
    "date": ISODate("2024-03-27"),
    "open": 2180.50,
    "high": 2195.30,
    "low": 2175.10,
    "close": 2190.80,
    "volume": 125000,
    "acquired_at": ISODate("2026-03-27T16:00:00")
}
```

In [28]:
def store_prices_to_mongo(df: pd.DataFrame, collection, asset_name: str) -> int:
    """
    Salva i dati OHLCV in MongoDB, evitando duplicati.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame con i dati OHLCV
    collection : pymongo.collection.Collection
        Collection MongoDB di destinazione
    asset_name : str
        Nome dell'asset (per log)
    
    Returns
    -------
    int
        Numero di documenti inseriti
    """
    if df.empty:
        return 0
    
    # Crea indice unico su (asset_name, date) per evitare duplicati
    collection.create_index(
        [("asset_name", 1), ("date", 1)],
        unique=True,
        name="idx_asset_date_unique"
    )
    
    # Converti DataFrame in lista di documenti
    acquired_at = datetime.utcnow()
    documents = []
    
    for _, row in df.iterrows():
        doc = {
            "asset_name":  row["asset_name"],
            "ticker":      row["ticker"],
            "date":        row["Date"],
            "open":        float(row["Open"]),
            "high":        float(row["High"]),
            "low":         float(row["Low"]),
            "close":       float(row["Close"]),
            "volume":      int(row["Volume"]) if pd.notna(row["Volume"]) else 0,
            "acquired_at": acquired_at,
        }
        documents.append(doc)
    
    # Insert con gestione duplicati (ordered=False salta i duplicati senza bloccarsi)
    inserted = 0
    try:
        result = collection.insert_many(documents, ordered=False)
        inserted = len(result.inserted_ids)
    except Exception as e:
        # BulkWriteError: alcuni documenti erano duplicati, gli altri sono stati inseriti
        if "duplicate key" in str(e).lower() or "bulkwriteerror" in str(type(e).__name__).lower():
            inserted = e.details.get("nInserted", 0) if hasattr(e, "details") else 0
            print(f"  {asset_name}: {inserted} nuovi inseriti, duplicati ignorati")
        else:
            raise
    
    return inserted

In [29]:
# Salva tutti i prezzi in MongoDB
print("=== SALVATAGGIO PREZZI IN MONGODB ===")
total_inserted = 0

for asset_name, df in all_prices.items():
    n = store_prices_to_mongo(df, prices_coll, asset_name)
    total_inserted += n
    print(f"  {asset_name}: {n} documenti inseriti")

print(f"\nTotale documenti nella collection 'prices': {prices_coll.count_documents({})}")

=== SALVATAGGIO PREZZI IN MONGODB ===
  gold: 0 nuovi inseriti, duplicati ignorati
  gold: 0 documenti inseriti
  silver: 0 nuovi inseriti, duplicati ignorati
  silver: 0 documenti inseriti
  dxy: 0 nuovi inseriti, duplicati ignorati
  dxy: 0 documenti inseriti
  sp500: 0 nuovi inseriti, duplicati ignorati
  sp500: 0 documenti inseriti

Totale documenti nella collection 'prices': 2014


In [30]:
# Verifica: query di esempio su MongoDB
print("=== VERIFICA DATI IN MONGODB ===")

# Query 1: ultimo prezzo dell'oro
last_gold = prices_coll.find_one(
    {"asset_name": "gold"},
    sort=[("date", -1)]
)
if last_gold:
    print(f"\nUltimo prezzo oro:")
    print(f"  Data:   {last_gold['date']}")
    print(f"  Close:  ${last_gold['close']:.2f}")
    print(f"  Volume: {last_gold['volume']:,}")

# Query 2: conteggio documenti per asset
print(f"\nDocumenti per asset:")
pipeline = [
    {"$group": {"_id": "$asset_name", "count": {"$sum": 1}}},
    {"$sort":  {"count": -1}}
]
for doc in prices_coll.aggregate(pipeline):
    print(f"  {doc['_id']}: {doc['count']} documenti")

=== VERIFICA DATI IN MONGODB ===

Ultimo prezzo oro:
  Data:   2026-03-27 00:00:00
  Close:  $4540.60
  Volume: 166,117

Documenti per asset:
  dxy: 504 documenti
  silver: 504 documenti
  gold: 504 documenti
  sp500: 502 documenti


## 4. Fonte 2 — Web Scraping (news finanziarie)

Usiamo `requests` + `BeautifulSoup` per scrappare headlines di news finanziarie.
Questa è la fonte **scraping** richiesta dalla FAQ 5.

**Target: Reuters Commodities** — pagina pubblica con headlines sulle materie prime.

In [31]:
def scrape_reuters_commodities() -> list:
    """
    Scrapa le headlines dalla sezione commodities di Reuters.
    
    Returns
    -------
    list[dict]
        Lista di dizionari con: title, url, source, scraped_at
    """
    url = "https://www.reuters.com/markets/commodities/"
    
    print(f"  Scraping: {url}")
    
    try:
        response = requests.get(url, headers=SCRAPING_HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"  ERRORE HTTP: {e}")
        return []
    
    soup = BeautifulSoup(response.text, "lxml")
    scraped_at = datetime.utcnow()
    articles = []
    
    # Reuters usa tag <a> con data-testid per i link degli articoli
    # Cerchiamo tutti i link che puntano ad articoli
    for link in soup.find_all("a", href=True):
        href = link.get("href", "")
        text = link.get_text(strip=True)
        
        # Filtra solo i link che sembrano articoli (contengono una data nel path)
        if ("/article/" in href or "/markets/" in href) and len(text) > 30:
            # Normalizza URL
            full_url = href if href.startswith("http") else f"https://www.reuters.com{href}"
            
            # Evita duplicati nella stessa sessione
            if not any(a["title"] == text for a in articles):
                articles.append({
                    "title":      text,
                    "url":        full_url,
                    "source":     "reuters",
                    "section":    "commodities",
                    "scraped_at": scraped_at,
                })
    
    print(f"  Trovate {len(articles)} headlines")
    return articles

In [32]:
def scrape_investing_rss() -> list:
    """
    Scrapa headlines finanziarie su oro da Investing.com RSS.
    Feed pubblico — nessuna autenticazione richiesta.
    """
    rss_url = "https://www.investing.com/rss/news_301.rss"  # Commodities
    
    print(f"  Scraping: Investing.com RSS (commodities)")
    
    try:
        response = requests.get(rss_url, headers=SCRAPING_HEADERS, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"  ERRORE HTTP: {e}")
        return []
    
    soup = BeautifulSoup(response.text, "lxml-xml")
    scraped_at = datetime.utcnow()
    articles = []
    
    for item in soup.find_all("item"):
        title_tag   = item.find("title")
        link_tag    = item.find("link")
        pubdate_tag = item.find("pubDate")
        
        if title_tag and link_tag:
            articles.append({
                "title":      title_tag.get_text(strip=True),
                "url":        link_tag.get_text(strip=True),
                "source":     "investing.com",
                "section":    "commodities",
                "published":  pubdate_tag.get_text(strip=True) if pubdate_tag else None,
                "scraped_at": scraped_at,
            })
    
    print(f"  Trovate {len(articles)} headlines")
    return articles

In [33]:
# Esegui scraping da tutte le fonti
print("=== SCRAPING NEWS FINANZIARIE ===")

all_news = []

# Fonte 1: Investing.com RSS (sostituisce Reuters — 401 Forbidden)
investing_news = scrape_investing_rss()
all_news.extend(investing_news)

# Fonte 2: Google News RSS
gnews = scrape_google_news_gold()
all_news.extend(gnews)

print(f"\nTotale headlines raccolte: {len(all_news)}")

=== SCRAPING NEWS FINANZIARIE ===
  Scraping: Investing.com RSS (commodities)
  Trovate 10 headlines
  Scraping: Google News RSS (gold price)
  Trovate 100 headlines

Totale headlines raccolte: 110


In [34]:
# Ispezione dei dati scrappati
if all_news:
    news_df = pd.DataFrame(all_news)
    print(f"Shape: {news_df.shape}")
    print(f"Colonne: {list(news_df.columns)}")
    print(f"\nDistribuzione per fonte:")
    print(news_df["source"].value_counts())
    print(f"\nEsempio headlines:")
    display(news_df[["title", "source"]].head(10))
else:
    print("Nessuna news raccolta — controlla la connessione internet del container")

Shape: (110, 6)
Colonne: ['title', 'url', 'source', 'section', 'published', 'scraped_at']

Distribuzione per fonte:
source
KITCO                                                    12
investing.com                                            10
AASTOCKS.com                                              9
Yahoo Finance UK                                          7
Mint                                                      5
Reuters                                                   4
The Economic Times                                        4
thestreet.com                                             3
Barron's                                                  3
TradingView                                               3
Investing.com                                             3
Goldman Sachs                                             3
J.P. Morgan                                               2
Business Insider                                          2
World Bank Blogs                     

,title,source
0,"Bitcoin set for weekly loss as Iran war, $14 b...",investing.com
1,Bitcoin set to end the week lower amid risk av...,investing.com
2,USDT0 Integrates With Tempo to Bring Omnichain...,investing.com
3,Bitcoin recovery depends on liquidity; US-Iran...,investing.com
4,Bitcoin falls below $70k as risk assets take a...,investing.com
5,BTCC Wins Most Secure Digital Asset Exchange b...,investing.com
6,T-REX Network and Zama Launch Institutional-Gr...,investing.com
7,Fannie Mae to accept crypto-backed mortgages f...,investing.com
8,Fannie Mae To Accept Crypto-backed Mortgage - WSJ,investing.com
9,Bitcoin ticks up above $71k as de-escalation h...,investing.com


### 4.1 Salvataggio news in MongoDB

In [35]:
def store_news_to_mongo(news_list: list, collection) -> int:
    """
    Salva le news in MongoDB, evitando duplicati basati su titolo e source.
    
    Parameters
    ----------
    news_list : list[dict]
        Lista di news da inserire
    collection : pymongo.collection.Collection
        Collection MongoDB di destinazione
    
    Returns
    -------
    int
        Numero di documenti inseriti
    """
    if not news_list:
        return 0
    
    # Indice unico su (title, source) per evitare duplicati
    collection.create_index(
        [("title", 1), ("source", 1)],
        unique=True,
        name="idx_title_source_unique"
    )
    
    inserted = 0
    try:
        result = collection.insert_many(news_list, ordered=False)
        inserted = len(result.inserted_ids)
    except Exception as e:
        if "duplicate key" in str(e).lower() or "bulkwriteerror" in str(type(e).__name__).lower():
            inserted = e.details.get("nInserted", 0) if hasattr(e, "details") else 0
            print(f"  {inserted} nuove news inserite, duplicati ignorati")
        else:
            raise
    
    return inserted

In [36]:
# Salva news in MongoDB
print("=== SALVATAGGIO NEWS IN MONGODB ===")
n_inserted = store_news_to_mongo(all_news, news_coll)
print(f"News inserite: {n_inserted}")
print(f"Totale documenti nella collection 'news': {news_coll.count_documents({})}")

=== SALVATAGGIO NEWS IN MONGODB ===
  19 nuove news inserite, duplicati ignorati
News inserite: 19
Totale documenti nella collection 'news': 156


## 5. Log di acquisizione

Registriamo un log di ogni sessione di acquisizione per tracciabilità e riproducibilità.

In [37]:
# Salva log di acquisizione
log_entry = {
    "timestamp":   datetime.utcnow(),
    "type":        "batch_historical",
    "assets":      list(ASSETS.keys()),
    "period":      HISTORICAL_PERIOD,
    "interval":    CANDLE_INTERVAL,
    "prices_total":  prices_coll.count_documents({}),
    "news_total":    news_coll.count_documents({}),
    "news_sources": ["reuters", "google_news"],
    "status":       "completed",
}

log_coll.insert_one(log_entry)
print("Log di acquisizione salvato")
print(json.dumps({k: str(v) for k, v in log_entry.items() if k != '_id'}, indent=2))

Log di acquisizione salvato
{
  "timestamp": "2026-03-28 15:32:55.886252",
  "type": "batch_historical",
  "assets": "['gold', 'silver', 'dxy', 'sp500']",
  "period": "2y",
  "interval": "1d",
  "prices_total": "2014",
  "news_total": "156",
  "news_sources": "['reuters', 'google_news']",
  "status": "completed"
}


## 6. Verifica finale — Query sui dati acquisiti

Due query di verifica sui dati (richieste dalla FAQ 6: almeno 2 query sul DBMS).

In [38]:
# QUERY 1: Prezzo medio di chiusura dell'oro per mese (ultimi 6 mesi)
print("=== QUERY 1: Prezzo medio chiusura ORO per mese (ultimi 6 mesi) ===")

six_months_ago = datetime.utcnow() - timedelta(days=180)

pipeline_monthly_avg = [
    {"$match": {
        "asset_name": "gold",
        "date": {"$gte": six_months_ago}
    }},
    {"$group": {
        "_id": {
            "year":  {"$year": "$date"},
            "month": {"$month": "$date"}
        },
        "avg_close": {"$avg": "$close"},
        "max_high":  {"$max": "$high"},
        "min_low":   {"$min": "$low"},
        "count":     {"$sum": 1}
    }},
    {"$sort": {"_id.year": 1, "_id.month": 1}}
]

for doc in prices_coll.aggregate(pipeline_monthly_avg):
    y, m = doc["_id"]["year"], doc["_id"]["month"]
    print(f"  {y}-{m:02d}: avg=${doc['avg_close']:.2f}, "
          f"range=[${doc['min_low']:.2f} — ${doc['max_high']:.2f}], "
          f"candele={doc['count']}")

=== QUERY 1: Prezzo medio chiusura ORO per mese (ultimi 6 mesi) ===
  2025-09: avg=$3840.80, range=[$3793.40 — $3865.50], candele=1
  2025-10: avg=$4044.37, range=[$3823.70 — $4358.00], candele=23
  2025-11: avg=$4082.12, range=[$3927.40 — $4228.70], candele=19
  2025-12: avg=$4311.06, range=[$4167.00 — $4556.30], candele=22
  2026-01: avg=$4730.86, range=[$4314.40 — $5586.20], candele=20
  2026-02: avg=$5011.04, range=[$4400.00 — $5280.00], candele=19
  2026-03: avg=$4885.78, range=[$4100.80 — $5405.00], candele=20


In [39]:
# QUERY 2: Top 10 giorni con massima variazione percentuale (oro)
print("=== QUERY 2: Top 10 giorni con massima variazione % giornaliera (ORO) ===")

pipeline_volatility = [
    {"$match": {"asset_name": "gold"}},
    {"$project": {
        "date":  1,
        "open":  1,
        "close": 1,
        "daily_change_pct": {
            "$multiply": [
                {"$divide": [
                    {"$subtract": ["$close", "$open"]},
                    "$open"
                ]},
                100
            ]
        }
    }},
    {"$sort": {"daily_change_pct": -1}},
    {"$limit": 10}
]

for doc in prices_coll.aggregate(pipeline_volatility):
    direction = "+" if doc["daily_change_pct"] > 0 else ""
    print(f"  {doc['date'].strftime('%Y-%m-%d')}: "
          f"{direction}{doc['daily_change_pct']:.2f}% "
          f"(open=${doc['open']:.2f} → close=${doc['close']:.2f})")

=== QUERY 2: Top 10 giorni con massima variazione % giornaliera (ORO) ===
  2026-02-03: +4.78% (open=$4680.00 → close=$4903.70)
  2026-02-06: +3.97% (open=$4762.00 → close=$4951.20)
  2026-03-27: +3.86% (open=$4371.80 → close=$4540.60)
  2025-04-09: +3.06% (open=$2965.80 → close=$3056.50)
  2025-04-16: +2.73% (open=$3238.30 → close=$3326.60)
  2025-04-10: +2.64% (open=$3073.90 → close=$3155.20)
  2026-01-22: +2.44% (open=$4791.90 → close=$4908.80)
  2026-02-18: +2.35% (open=$4872.20 → close=$4986.50)
  2025-10-13: +2.31% (open=$4016.00 → close=$4108.60)
  2025-06-02: +2.24% (open=$3296.90 → close=$3370.60)


## 7. Riepilogo

Questa fase ha coperto:

| Fase | Fatto |
|---|---|
| Fonte 1 — API | Dati OHLCV da Yahoo Finance per oro, argento, DXY, S&P500 |
| Fonte 2 — Scraping | Headlines da Reuters + Google News RSS |
| Storage | Tutti i dati grezzi salvati in MongoDB (collection `prices` e `news`) |
| Query | 2 query di verifica (media mensile, top volatilità) |
| Tracciabilità | Log di acquisizione salvato |

**Prossimo notebook:** `02_data_profiling.ipynb` — analisi qualità dei dati acquisiti.

In [40]:
# Riepilogo finale
print("=" * 60)
print("RIEPILOGO ACQUISIZIONE")
print("=" * 60)
print(f"Database:    {MONGO_DB}")
print(f"MongoDB URI: {MONGO_URI}")
print(f"")
print(f"Collection 'prices':")
print(f"  Documenti totali: {prices_coll.count_documents({})}")
for asset in ASSETS:
    n = prices_coll.count_documents({"asset_name": asset})
    print(f"  - {asset}: {n} candele")
print(f"")
print(f"Collection 'news':")
print(f"  Documenti totali: {news_coll.count_documents({})}")
print(f"")
print(f"Collection 'acquisition_log':")
print(f"  Log registrati: {log_coll.count_documents({})}")
print("=" * 60)

RIEPILOGO ACQUISIZIONE
Database:    finpulse
MongoDB URI: mongodb://mongo:27017

Collection 'prices':
  Documenti totali: 2014
  - gold: 504 candele
  - silver: 504 candele
  - dxy: 504 candele
  - sp500: 502 candele

Collection 'news':
  Documenti totali: 156

Collection 'acquisition_log':
  Log registrati: 4
